# Transfer experiments

> Pretrain the CNN on OSSL, train and score zero-shot, head-only, fine-tuned and Fukushima-only models over several seeds, and check a model within a target range

In [ ]:
#| default_exp transfer

In [ ]:
#| export
import copy
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from fastai.layers import NormType
import fastai.callback.schedule
from fastai.torch_core import set_seed
from soilspectfm.preprocessing import SNV, SavitzkyGolay
from soilspecdl.all import get_spectral_dls, get_splitter, spectral_learner, SpectralCNN, BatchSNV
from ossl_fuku_transfer.data import data_path, shared_wns

In [ ]:
from fastcore.test import *

`03_kex` and `04_cec` use this module on real data. The examples here use small random arrays, so they run without the data.

## Models and preprocessing

In [ ]:
#| export
def preprocessings()->dict: # Name: unfitted scikit-learn pipeline
    "SNV, and SNV followed by a Savitzky-Golay first derivative"
    return {'snv': Pipeline([('snv', SNV())]),
            'snv_d1': Pipeline([('snv', SNV()), ('deriv', SavitzkyGolay(deriv=1))])}

def new_model()->SpectralCNN:
    "The CNN of Albinet et al. (2022), with random weights"
    return SpectralCNN(nfs=(16, 32, 64, 128, 256), pool_sz=3, norm_type=NormType.Batch)

Each preprocessing is fitted on OSSL and applied unchanged to Fukushima. In every model, `BatchSNV` then standardizes each spectrum in a batch. The CNN takes a batch of spectra and returns one prediction per spectrum:

In [ ]:
test_eq(new_model()(torch.randn(4, 1, len(shared_wns))).shape, (4,))
list(preprocessings())

## Pretraining

In [ ]:
#| export
def model_dir()->Path: # Folder of pretrained weights and training histories
    "`models/ossl-fuku-transfer` in `data_path()`"
    p = data_path()/'models'/'ossl-fuku-transfer'
    p.mkdir(parents=True, exist_ok=True)
    return p

def pretrain(
    X, # Preprocessed OSSL spectra
    y, # Targets, such as `log1p` of a soil property
    fname, # File name for the weights, without extension
    seed=42, # Seed for the validation split and the training
    epochs=20, # Number of one-cycle epochs
    lr=1e-3, # Maximum learning rate
)->SpectralCNN: # The trained model, on the CPU
    "Train `new_model` on OSSL, or load the weights that an earlier run saved under `fname`"
    fn, model = model_dir()/f'{fname}.pth', new_model()
    if not fn.exists():
        set_seed(seed)
        dls, _ = get_spectral_dls(X, y, shared_wns, splitter=get_splitter(0.9, 0.1, seed=seed), batch_tfms=[BatchSNV()])
        learn = spectral_learner(dls, model)
        with learn.no_bar(), learn.no_logging(): learn.fit_one_cycle(epochs, lr_max=lr)
        torch.save(model.state_dict(), fn)
        hist = pd.DataFrame(learn.recorder.values, columns=learn.recorder.metric_names[1:-1])
        hist.to_csv(fn.with_suffix('.csv'), index_label='epoch')
    model.load_state_dict(torch.load(fn, map_location='cpu'))
    return model

def pretrain_history(fname)->pd.DataFrame: # One row per epoch: losses and validation R²
    "The training history that `pretrain` saved under `fname`"
    return pd.read_csv(model_dir()/f'{fname}.csv', index_col='epoch')

`pretrain` splits OSSL at random into 90% training and 10% validation, and trains for `epochs` one-cycle epochs, as in the soilspecdl quick start. It saves the weights and the training history in `model_dir()`. A later call with the same `fname` loads them instead of training again. Delete the files to train again. Training on an Apple silicon GPU (MPS) is not fully deterministic, even with a fixed seed.

In [ ]:
#| export
def range_check(
    X, # Preprocessed OSSL spectra, as passed to `pretrain`
    y, # Targets, as passed to `pretrain`
    model, # Model returned by `pretrain`
    lo, # Lower bound of the range, in the units of `y`
    hi, # Upper bound of the range
    seed=42, # Seed passed to `pretrain`
)->dict: # Sample count, R² and correlation, for all validation samples and those in range
    "Score `model` on its OSSL validation samples: all of them, and those with `y` between `lo` and `hi`"
    dls, _ = get_spectral_dls(X, y, shared_wns, splitter=get_splitter(0.9, 0.1, seed=seed), batch_tfms=[BatchSNV()])
    learn = spectral_learner(dls, copy.deepcopy(model))
    with learn.no_bar(): p, t = learn.get_preds(ds_idx=1)
    p, t = p.numpy().ravel(), t.numpy().ravel()
    m = (t >= lo) & (t <= hi)
    return {'n all': len(t), 'R² all': r2_score(t, p), 'r all': np.corrcoef(t, p)[0, 1],
            'n in range': int(m.sum()), 'R² in range': r2_score(t[m], p[m]), 'r in range': np.corrcoef(t[m], p[m])[0, 1]}

A model scored over a narrow range has a lower R² than over a wide one, because the targets vary less. `range_check` measures a pretrained model on the part of its OSSL validation set that covers the Fukushima range. This gives the reference for Fukushima results.

## Fukushima models

In [ ]:
#| export
def fuku_dls(
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seed=0, # Seed for choosing the validation samples
    valid_pct=0.1, # Share of the training samples used for validation
):
    "`DataLoaders` over all Fukushima spectra, with the test set indices in `test_idx`"
    train_idx, test_idx = np.flatnonzero(~is_test), np.flatnonzero(is_test)
    np.random.RandomState(seed).shuffle(train_idx)
    n_valid = int(len(train_idx) * valid_pct)
    split = lambda X, y: (train_idx[n_valid:], train_idx[:n_valid], test_idx)
    dls = get_spectral_dls(X, y, shared_wns, splitter=split, batch_tfms=[BatchSNV()])[0]
    dls.test_idx = test_idx
    return dls

def test_preds(learn)->np.ndarray: # One prediction per test sample
    "Predict the test set of `learn.dls`"
    with learn.no_bar(): p, _ = learn.get_preds(dl=learn.dls.test_dl(learn.dls.test_idx.tolist(), with_labels=True))
    return p.numpy().ravel()

With 100 samples of which 20 are test samples, 8 of the other 80 go to validation:

In [ ]:
rng = np.random.default_rng(0)
X_syn, y_syn = rng.random((100, len(shared_wns))), rng.random(100)
is_test_syn = np.arange(100) >= 80
dls_syn = fuku_dls(X_syn, y_syn, is_test_syn)
test_eq((len(dls_syn.train_ds), len(dls_syn.valid_ds), len(dls_syn.test_idx)), (72, 8, 20))

In [ ]:
#| export
def fit_models(
    pretrained, # Model returned by `pretrain`
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seed=0, # Seed for the validation split and the training
    epochs=20, # Training epochs after the first, frozen, epoch of fine-tuning
    lr=1e-3, # Maximum learning rate
)->dict: # Model name: test predictions
    "Test predictions of the zero-shot, head-only, fine-tuned and Fukushima-only models"
    dls, res = fuku_dls(X, y, is_test, seed), {}
    def _fit(learn, fit):
        with learn.no_bar(), learn.no_logging(): fit(learn)
        return test_preds(learn)
    res['zero-shot'] = test_preds(spectral_learner(dls, copy.deepcopy(pretrained)))
    set_seed(seed)
    learn = spectral_learner(dls, copy.deepcopy(pretrained), train_bn=False)
    learn.freeze()
    res['head only'] = _fit(learn, lambda l: l.fit_one_cycle(epochs + 1, lr_max=lr))
    set_seed(seed)
    res['fine-tuned'] = _fit(spectral_learner(dls, copy.deepcopy(pretrained)), lambda l: l.fine_tune(epochs, base_lr=lr))
    set_seed(seed)
    res['Fukushima only'] = _fit(spectral_learner(dls, new_model()), lambda l: l.fit_one_cycle(epochs + 1, lr_max=lr))
    return res

The four models see the same validation split and the same number of epochs:

- Zero-shot predicts with the pretrained model as it is.
- Head only trains the pretrained head, a linear layer of 257 weights, for `epochs + 1` epochs. The backbone weights stay fixed, including the BatchNorm scale and shift. The BatchNorm running statistics still update to the Fukushima batches during training.
- Fine-tuned uses fastai's `fine_tune`: one epoch with the backbone frozen at learning rate `lr`, then `epochs` epochs of the whole model. In that second phase the head's maximum learning rate is `lr / 2`, and the backbone's is `lr / 200`. With `lr=1e-3`, the backbone trains at most at 5e-6, 200 times lower than the Fukushima-only model, and stays close to its pretrained weights.
- Fukushima only trains `new_model` from random weights for `epochs + 1` epochs.

With an untrained model in place of a pretrained one and one epoch, each model predicts the 20 test samples:

In [ ]:
preds_syn = fit_models(new_model(), X_syn, y_syn, is_test_syn, epochs=1)
test_eq({k: v.shape for k, v in preds_syn.items()}, {k: (20,) for k in ['zero-shot', 'head only', 'fine-tuned', 'Fukushima only']})

## Scores

In [ ]:
#| export
def scores(
    y, # Observed targets, as `log1p` of the property
    p, # Predicted targets
)->dict: # R² and correlation of the targets, RMSE and bias of the property
    "R² and correlation `r` of `log1p` targets, and the RMSE and bias of the property itself"
    err = np.expm1(p) - np.expm1(y)
    return {'R²': r2_score(y, p), 'r': np.corrcoef(y, p)[0, 1], 'RMSE': np.sqrt((err**2).mean()), 'bias': err.mean()}

A prediction that adds a constant to every `log1p` target keeps a correlation of 1, but loses R², and gains a bias:

In [ ]:
s = scores(y_syn, y_syn + 0.2)
test_close(s['r'], 1)
assert s['R²'] < 1 and s['bias'] > 0
{k: round(v, 3) for k, v in s.items()}

In [ ]:
#| export
def compare(
    pretrained, # Preprocessing name: model returned by `pretrain`
    X, # Preprocessing name: preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seeds=range(5), # Seeds to repeat `fit_models` with
    **kwargs, # Passed to `fit_models`
)->tuple: # (scores, one row per preprocessing, model and seed; predictions by (preprocessing, model, seed))
    "Run `fit_models` for each preprocessing and seed, and score the test predictions"
    rows, preds = [], {}
    for name, model in pretrained.items():
        for seed in seeds:
            for m, p in fit_models(model, X[name], y, is_test, seed, **kwargs).items():
                rows.append({'preprocessing': name, 'model': m, 'seed': seed, **scores(y[is_test], p)})
                preds[name, m, seed] = p
    return pd.DataFrame(rows), preds

def summarise(res:pd.DataFrame)->pd.DataFrame: # Mean and standard deviation over seeds
    "Mean and standard deviation of each score over the seeds, by preprocessing and model"
    order = ['zero-shot', 'head only', 'fine-tuned', 'Fukushima only']
    out = res.groupby(['preprocessing', 'model'])[['R²', 'r', 'RMSE', 'bias']].agg(['mean', 'std'])
    return out.reindex(order, level='model')

`compare` repeats the Fukushima-side training over several seeds. The seed changes the validation split and the initial weights of the new heads and models. The pretrained models stay the same. `summarise` gives the mean and standard deviation of each score:

In [ ]:
res_syn, _ = compare({'snv': new_model()}, {'snv': X_syn}, y_syn, is_test_syn, seeds=range(2), epochs=1)
test_eq(len(res_syn), 2 * 4)
summarise(res_syn).shape

## Plots and error analysis

In [ ]:
#| export
def plot_preds(
    preds, # Predictions returned by `compare`
    y, # Observed test targets
    seed=0, # Seed whose predictions to plot
    label='log1p(target)', # Axis label
):
    "Predicted against observed test targets, one panel per preprocessing and model"
    names = list(dict.fromkeys(k[0] for k in preds))
    models = list(dict.fromkeys(k[1] for k in preds))
    fig, axs = plt.subplots(len(names), len(models), figsize=(3 * len(models), 3 * len(names)), sharex=True, sharey=True, squeeze=False)
    lims = [y.min(), y.max()]
    for row, n in zip(axs, names):
        for ax, m in zip(row, models):
            p = preds[n, m, seed]
            ax.scatter(y, p, s=5, alpha=0.5)
            ax.plot(lims, lims, c='k', lw=0.8)
            ax.set_title(f'{n}, {m}: R² {r2_score(y, p):.2f}', fontsize=9)
    for ax in axs[-1]: ax.set_xlabel(f'Observed {label}')
    for ax in axs[:, 0]: ax.set_ylabel(f'Predicted {label}')
    return fig

In [ ]:
plot_preds(compare({'snv': new_model()}, {'snv': X_syn}, y_syn, is_test_syn, seeds=range(1), epochs=1)[1], y_syn[is_test_syn]);

In [ ]:
#| export
def error_table(
    preds, # Predictions returned by `compare`
    y, # Observed test targets
    labels, # Group label of each test sample, such as its soil type
    name, # Preprocessing to report
    models=('head only', 'fine-tuned', 'Fukushima only'), # Models to report
)->pd.DataFrame: # One row per group: mean absolute error of each model, and sample count
    "Mean absolute error of the seed-averaged predictions of each model, by group"
    seeds = sorted({k[2] for k in preds if k[0] == name})
    err = {m: np.abs(np.mean([preds[name, m, s] for s in seeds], axis=0) - y) for m in models}
    return pd.DataFrame(err).assign(n=1).groupby(np.asarray(labels)).agg({**{m: 'mean' for m in models}, 'n': 'sum'})

`error_table` averages each model's predictions over the seeds, then reports the mean absolute error of each group:

In [ ]:
tab = error_table(compare({'snv': new_model()}, {'snv': X_syn}, y_syn, is_test_syn, seeds=range(2), epochs=1)[1], y_syn[is_test_syn], np.repeat(['a', 'b'], 10), 'snv')
test_eq(tab.n.tolist(), [10, 10])
tab.round(3)